# Sanity Check: FlexLMKDTrainer

Smoke test for the OpenWebText + GPT-2 teacher experiment before submitting to Snellius.

Checks:
1. OpenWebText loader returns correct tensor shapes
2. Teacher loads and produces logits of the right shape
3. CE and KL losses are both present and finite
4. Level 2 CE loss starts low (~3.5) due to GPT-2 init
5. KL loss is non-zero (teacher signal is flowing)
6. All three levels log independently

In [1]:
import sys
sys.path.insert(0, '.')  # run from repo root

import torch
import torch.nn.functional as F

## 1. Data loader (WikiText-103 used as stand-in — content doesn't matter for shape checks)

In [2]:
import importlib
import utils
importlib.reload(utils)

train, val, test = utils.load_wikitext(max_seq_length=64, batch_size=2)
batch = next(iter(train))
input_ids, labels = batch

assert input_ids.shape == (2, 64), f"Expected (2, 64), got {input_ids.shape}"
assert (input_ids == labels).all(), "input_ids and labels should be identical"
print(f"Loader OK — batch shape: {input_ids.shape}, dtype: {input_ids.dtype}")

/home/jort/Desktop/thesis/FlexViT/myenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Map: 100%|██████████| 3760/3760 [00:00<00:00, 9086.34 examples/s]


Loader OK — batch shape: torch.Size([2, 64]), dtype: torch.int64


## 2. Teacher loads and produces correct output shape

In [3]:
from transformers import GPT2LMHeadModel

teacher = GPT2LMHeadModel.from_pretrained("gpt2")
teacher.eval()
for p in teacher.parameters():
    p.requires_grad_(False)

with torch.no_grad():
    teacher_out = teacher(input_ids)

assert teacher_out.logits.shape == (2, 64, 50257), f"Unexpected shape: {teacher_out.logits.shape}"
print(f"Teacher OK — logits shape: {teacher_out.logits.shape}")

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 12117.77it/s]


Teacher OK — logits shape: torch.Size([2, 64, 50257])


## 3. FlexGPT forward pass at each level

In [4]:
from networks.flexgpt import FlexGPT, FlexGPTConfig
import utils as utils_module

cfg = FlexGPTConfig(
    vocab_size=50257,
    max_seq_length=64,
    num_layers=2,          # small for speed
    hidden_dims=(384, 512, 768),
    num_heads=(6, 8, 12),
    mlp_dims=(1536, 2048, 3072),
    dropout=0.0,
    pretrained_hf_model=None,  # random init for shape check
)
model = cfg.make_model()

for level in range(model.max_level() + 1):
    model.set_level_use(level)
    logits = model(input_ids)
    assert logits.shape == (2, 64, 50257), f"Level {level}: unexpected shape {logits.shape}"
    print(f"Level {level} forward OK — logits shape: {logits.shape}")

Level 0 forward OK — logits shape: torch.Size([2, 64, 50257])
Level 1 forward OK — logits shape: torch.Size([2, 64, 50257])
Level 2 forward OK — logits shape: torch.Size([2, 64, 50257])


## 4. CE and KL loss values

In [5]:
T = 2.0
vocab_size = 50257

teacher_logits = teacher_out.logits  # [2, 64, V]

print(f"{'Level':<8} {'CE loss':<12} {'KL loss':<12} {'PPL':<10}")
print("-" * 44)

for level in range(model.max_level() + 1):
    model.set_level_use(level)
    with torch.no_grad():
        logits = model(input_ids)

    student = logits[:, :-1]           # [2, 63, V]
    teacher = teacher_logits[:, :-1]   # [2, 63, V]
    targets = input_ids[:, 1:].reshape(-1)

    ce_loss = F.cross_entropy(student.reshape(-1, vocab_size), targets)

    kl_loss = F.kl_div(
        F.log_softmax(student / T, dim=-1).reshape(-1, vocab_size),
        F.softmax(teacher / T, dim=-1).reshape(-1, vocab_size),
        reduction="batchmean",
    ) * (T ** 2)

    ppl = torch.exp(ce_loss)

    assert torch.isfinite(ce_loss), f"Level {level}: CE loss is not finite"
    assert torch.isfinite(kl_loss), f"Level {level}: KL loss is not finite"
    assert kl_loss > 0, f"Level {level}: KL loss is zero — teacher signal not flowing"

    print(f"{level:<8} {ce_loss.item():<12.4f} {kl_loss.item():<12.4f} {ppl.item():<10.2f}")

Level    CE loss      KL loss      PPL       
--------------------------------------------
0        82.3748      129.9389     595541261635190744518931246586265600.00
1        93.2551      153.2155     inf       
2        115.2998     196.7394     inf       


## 5. Level 2 CE loss after GPT-2 weight loading

With pretrained weights, level 2 CE loss should start around 3.5 (GPT-2 Small baseline).
Random init gives ~10.8 (log(50257)).

In [8]:
cfg_pretrained = FlexGPTConfig(
    vocab_size=50257,
    max_seq_length=1024,
    num_layers=12,
    hidden_dims=(384, 512, 768),
    num_heads=(6, 8, 12),
    mlp_dims=(1536, 2048, 3072),
    dropout=0.0,
)
model_pretrained = cfg_pretrained.make_model()

# pretrained_hf_model in the config is only read by the trainer — load weights explicitly here
utils.load_gpt2_weights_into_flexgpt(model_pretrained, "gpt2")
model_pretrained.eval()

# Fresh batch at seq_len=1024 to match GPT-2's positional embeddings
train_1024, _, _ = utils.load_wikitext(max_seq_length=1024, batch_size=2)
input_ids_1024, _ = next(iter(train_1024))

model_pretrained.set_level_use(2)
with torch.no_grad():
    logits = model_pretrained(input_ids_1024)

ce = F.cross_entropy(
    logits[:, :-1].reshape(-1, vocab_size),
    input_ids_1024[:, 1:].reshape(-1),
)
print(f"Level 2 CE loss (pretrained): {ce.item():.4f}")
print(f"Level 2 PPL    (pretrained): {torch.exp(ce).item():.2f}")
assert ce.item() < 6.0, f"CE loss {ce.item():.2f} too high — GPT-2 weights may not have loaded"
print("GPT-2 weight loading OK")

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 17266.27it/s]


Level 2 CE loss (pretrained): 3.7713
Level 2 PPL    (pretrained): 43.44
GPT-2 weight loading OK


## 6. One training step end-to-end

Verifies the teacher loads, gradients flow, and all level losses are finite.
Runs the step directly without Lightning (DDP is incompatible with notebooks).

In [11]:
from training import FlexLMKDTrainer
from config.experiments import FlexLMKDTrainingContext
from transformers import GPT2LMHeadModel

cfg_small = FlexGPTConfig(
    vocab_size=50257,
    max_seq_length=64,
    num_layers=2,
    hidden_dims=(384, 512, 768),
    num_heads=(6, 8, 12),
    mlp_dims=(1536, 2048, 3072),
    dropout=0.0,
    pretrained_hf_model=None,
)
student = cfg_small.make_model()
student.train()

teacher = GPT2LMHeadModel.from_pretrained("gpt2")
teacher.eval()
for p in teacher.parameters():
    p.requires_grad_(False)

optimizer = torch.optim.AdamW(student.parameters(), lr=3e-4)
kd_lambda = 0.5
T = 2.0
V = 50257

# One manual training step
optimizer.zero_grad()
total_loss = 0.0

with torch.no_grad():
    teacher_logits = teacher(input_ids).logits  # input_ids from cell 3

for level in range(student.max_level() + 1):
    student.set_level_use(level)
    logits = student(input_ids)

    s = logits[:, :-1]
    t = teacher_logits[:, :-1]
    targets = input_ids[:, 1:].reshape(-1)

    ce  = F.cross_entropy(s.reshape(-1, V), targets)
    kl  = F.kl_div(F.log_softmax(s / T, dim=-1).reshape(-1, V),
                   F.softmax(t / T, dim=-1).reshape(-1, V),
                   reduction="batchmean") * T**2
    loss = (1 - kd_lambda) * ce + kd_lambda * kl
    loss.backward()

    assert torch.isfinite(loss), f"Level {level}: loss is not finite"
    print(f"Level {level} — CE: {ce.item():.4f}  KL: {kl.item():.4f}  loss: {loss.item():.4f}")
    total_loss += loss.detach()

optimizer.step()
print(f"\nTotal loss: {total_loss.item():.4f}")
print("Training step OK — gradients flowed through all levels")

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 18356.90it/s]


Level 0 — CE: 80.1270  KL: 130.5868  loss: 105.3569
Level 1 — CE: 97.4417  KL: 157.0695  loss: 127.2556
Level 2 — CE: 109.7868  KL: 197.4608  loss: 153.6238

Total loss: 386.2363
Training step OK — gradients flowed through all levels
